In [1]:
import pandas as pd
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [2]:
import mlflow
# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Tech-Challenge-Telco-Churn")

c:\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location=('file:///c:/Users/lara-/OneDrive/Desktop/POS TECH '
 'ML/projeto-tech-challenge-churn/notebooks/mlruns/2'), creation_time=1777127798390, experiment_id='2', last_update_time=1777127798390, lifecycle_stage='active', name='Tech-Challenge-Telco-Churn', tags={'phase': 'baseline/test'}, workspace='default'>

In [4]:
# # Define o banco de dados na raiz do projeto
# mlflow.set_tracking_uri("sqlite:///../mlflow.db")
# # Cria ou seleciona um experimento com nome específico
# mlflow.set_experiment("Projeto_Churn_Tech_Challenge")

# --------------- REGRESSÃO LOGISTICA ---------------

In [3]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.data
import matplotlib.pyplot as plt
import os
import json
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature

csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(
    df, source=csv_path, name="dataset_normalizado_enginereed")

# --- 1. CONFIGURAÇÕES E CUSTOS (Ticket Médio R$ 65) ---
test_size = 0.2
random_state = 42
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65,00
COST_FN = TICKET_MEDIO * 5    # R$ 325,00

# Cria a pasta de saída para os artefatos locais antes do upload
os.makedirs("outputs", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(),
         X_train.select_dtypes(include=['int64', 'float64']).columns)
    ]
)

# Fit e transformando para DataFrames (melhor para as signatures do MLflow)
X_train_df = pd.DataFrame(preprocessador.fit_transform(X_train),
                          columns=preprocessador.get_feature_names_out())
X_test_df = pd.DataFrame(preprocessador.transform(X_test),
                         columns=preprocessador.get_feature_names_out())

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Regressao_Logistica_v4") as run:

    model = LogisticRegression(random_state=random_state, max_iter=1000)
    mlflow.log_input(dataset, context="training/test")
    model.fit(X_train_df, y_train)

    # Predições e Probabilidades
    y_probs = model.predict_proba(X_test_df)[:, 1]
    threshold = 0.50  # Threshold ajustado para priorizar menor FN de Churn
    y_pred = (y_probs >= threshold).astype(int)

    # Calculando as métricas solicitadas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    cm = confusion_matrix(y_test, y_pred)
    total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

    # Log de Tags, Parâmetros e Metrics
    mlflow.set_tags({
        "model_type": "regressao_logistica",
        "model_architecture": "regressao-logistica",
        "framework": "sklearn",
        "phase": "baseline",
        "dataset_name": "dataset_normalizado_enginereed",
        "dataset_size": "7043",
        "feature_set": "v2-clear-engineered"
    })

    mlflow.log_params({
        "model.model_type": "LogisticRegression",
        "model.max_iter": 1000,
        "data.random_state": random_state,
        "data.test_size": test_size,
        "data.stratify": True,
        "train.train_samples": len(X_train),
        "test.test_samples": len(X_test),
        "threshold.value": threshold,
        "business.ticket_medio": TICKET_MEDIO,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='darkorange',
                lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC (Receiver Operating Characteristic)")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate (Recall)")
    ax_roc.legend(loc="lower right")
    fig_roc.savefig("outputs/rl/roc_curve.png")
    mlflow.log_artifact("outputs/rl/roc_curve.png", artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[
                           'Fica', 'Churn']).plot(ax=ax_cm, cmap='Blues')
    ax_cm.set_title(f"Matriz de Confusão (Threshold {threshold})")
    fig_cm.savefig("outputs/rl/confusion_matrix.png")
    mlflow.log_artifact("outputs/rl/confusion_matrix.png",
                        artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: CLASSIFICATION REPORT ---
    report_path = "outputs/rl/classification_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Classification Report - Threshold: {threshold}\n")
        f.write("-" * 50 + "\n")
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact(report_path, artifact_path="diagnostics")

    # --- ARTEFATO 4: PREPROCESSADOR E METADADOS ---
    joblib.dump(preprocessador, "outputs/rl/preprocessador.pkl")
    mlflow.log_artifact("outputs/rl/preprocessador.pkl",
                        artifact_path="preprocessor")

    with open("outputs/rl/feature_names.json", "w") as f:
        json.dump(list(X_train_df.columns), f)
    mlflow.log_artifact("outputs/rl/feature_names.json", artifact_path="data")

   # 5. Log do Modelo PyTorch
    signature = infer_signature(X_test.head(5), y_pred[:5])

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="modelo-regressao-logistica",
        signature=signature,
        registered_model_name="regressao-logistica-classifier-churn"
    )

    # --- MODEL CARD (SE EXISTIR) ---
    if os.path.exists("docs/model-card.md"):
        mlflow.log_artifact("docs/model-card.md", artifact_path="model_card")

print(f"Finalizado! AUC-ROC: {roc_auc:.4f} | Custo Total: R${total_cost:.2f}")


c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Python311\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-in

Finalizado! AUC-ROC: 0.8444 | Custo Total: R$59865.00


Registered model 'regressao-logistica-classifier-churn' already exists. Creating a new version of this model...
Created version '5' of model 'regressao-logistica-classifier-churn'.


# --------------- DUMMY CLASSIFIER ---------------

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
import os
import json
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, accuracy_score,
                             f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, auc, classification_report)
from mlflow.models.signature import infer_signature


csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(
    df, source=csv_path, name="dataset_normalizado_enginereed")


# --- 1. CONFIGURAÇÕES E CUSTOS (Alinhado com o Baseline) ---
SEED = 20
test_size = 0.2
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325
THRESHOLD = 0.40             # Threshold definido pelo usuário

os.makedirs("outputs/dummy", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(), X_train.select_dtypes(
            include=['int64', 'float64']).columns)
    ]
)

# Transformação para o fit
X_train_transformed = preprocessador.fit_transform(X_train)
X_test_transformed = preprocessador.transform(X_test)
feature_names = preprocessador.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)


# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Dummy_Classifier_baseline_v2"):
   
    # Modelo Dummy
    modelo_dummy = DummyClassifier(strategy='stratified', random_state=SEED)
    mlflow.log_input(dataset, context="training/test")
    modelo_dummy.fit(X_train_df, y_train)

    # Predições (Dummy também usa threshold de 0.5 por padrão na probabilidade)
    y_probs = modelo_dummy.predict_proba(X_test_df)[:, 1]
    y_pred = (y_probs >= THRESHOLD).astype(int)

    # Cálculo de Métricas Técnicas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    # Cálculo de Custo de Negócio
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    total_cost = (fp * COST_FP) + (fn * COST_FN)

    # Log de Parâmetros e Tags
    # Log de Tags
    mlflow.set_tags({
        "model_type": "dummy_classifier",
        "model_architecture": "dummy_classifier",
        "framework": "sklearn",
        "phase": "baseline",
        "dataset_name": "dataset_normalizado_enginereed",
        "dataset_size": "7043",
        "feature_set": "v2-clear-engineered"
    })
    
    mlflow.log_params({
        "model.model_type": "DummyClassifier",
        "data.random_state": SEED,
        "data.test_size": test_size,
        "data.stratify": True,
        "train.train_samples": len(X_train),
        "test.test_samples": len(X_test),
        "threshold.value": THRESHOLD,
        "business.ticket_medio": TICKET_MEDIO,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

        # Log de Métricas
    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })
   
    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='gray', lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC - Dummy Classifier")
    ax_roc.legend()
    fig_roc.savefig("outputs/dummy/roc_curve_dummy.png")
    mlflow.log_artifact("outputs/dummy/roc_curve_dummy.png",
                        artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[
                           'Fica', 'Churn']).plot(ax=ax_cm, cmap='Greys')
    ax_cm.set_title('Matriz de Confusão - Dummy Classifier')
    fig_cm.savefig("outputs/dummy/confusion_matrix_dummy.png")
    mlflow.log_artifact(
        "outputs/dummy/confusion_matrix_dummy.png", artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: CLASSIFICATION REPORT ---
    report_path = "outputs/dummy/classification_report_dummy.txt"
    with open(report_path, "w") as f:
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact(report_path, artifact_path="diagnostics")

    # --- ARTEFATO 4: PREPROCESSADOR E DATA ---
    joblib.dump(preprocessador, "outputs/dummy/preprocessador.pkl")
    mlflow.log_artifact("outputs/dummy/preprocessador.pkl",
                        artifact_path="preprocessor")

    with open("outputs/dummy/feature_names.json", "w") as f:
        json.dump(list(feature_names), f)
    mlflow.log_artifact("outputs/dummy/feature_names.json",
                        artifact_path="data")

    # --- SALVAR MODELO COM SIGNATURE ---
     # 5. Log do Modelo PyTorch
    signature = infer_signature(X_test.head(5), y_pred[:5])

    mlflow.sklearn.log_model(
        sk_model=modelo_dummy,
        artifact_path="modelo-dummy-classifier",
        signature=signature,
        registered_model_name="dummy-classifier-churn"
    )
    print(f"===== Dummy Classifier Gravado! Custo Negócio: R$ {total_cost:.2f} =====")


# --------------- ÁRVORE DE DECISÃO ---------------

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature

# --- 1. CONFIGURAÇÕES E CUSTOS ---
SEED = 42
test_size = 0.2
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325
max_depth = 5 

os.makedirs("outputs/decision_tree", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=SEED, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(), X_train.select_dtypes(include=['int64', 'float64']).columns)
    ]
)

X_train_df = pd.DataFrame(preprocessador.fit_transform(X_train), columns=preprocessador.get_feature_names_out())
X_test_df = pd.DataFrame(preprocessador.transform(X_test), columns=preprocessador.get_feature_names_out())

# Dataset para o MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="dataset_normalizado_enginereed")

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Decision_Tree_Baseline_v4"):
    mlflow.log_input(dataset, context="training/test")
    model_dt = DecisionTreeClassifier(max_depth=max_depth, random_state=SEED)
    model_dt.fit(X_train_df, y_train)

    # Predições com Threshold Ajustado (0.30) para diminuir Falso Negativo
    y_probs = model_dt.predict_proba(X_test_df)[:, 1]
    threshold = 0.50
    y_pred = (y_probs >= threshold).astype(int)

    # Cálculos de Métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)
    
    cm = confusion_matrix(y_test, y_pred)
    total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

    
    # Log de Tags
    mlflow.set_tags({
        "model_type": "arvore_de_decisao",
        "model_architecture": "arvore-de-decisao",
        "framework": "sklearn",
        "phase": "baseline",
        "dataset_name": "dataset_normalizado_enginereed",
        "dataset_size": "7043",
        "feature_set": "v2-clear-engineered"
    })

    mlflow.log_params({
        "model.model_type": "DecisionTreeClassifier",
        "model.max_depth": max_depth,
        "data.random_state": SEED,
        "data.test_size": test_size,
        "data.stratify": True,
        "train.train_samples": len(X_train),
        "test.test_samples": len(X_test),
        "threshold.value": threshold,
        "business.ticket_medio": TICKET_MEDIO,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })


    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='seagreen', lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC - Decision Tree")
    ax_roc.legend()
    fig_roc.savefig("outputs/decision_tree/roc_curve_dt.png")
    mlflow.log_artifact("outputs/decision_tree/roc_curve_dt.png", artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fica', 'Churn']).plot(ax=ax_cm, cmap='Greens')
    ax_cm.set_title(f'Matriz de Confusão DT (Threshold {threshold})')
    fig_cm.savefig("outputs/decision_tree/confusion_matrix_dt.png")
    mlflow.log_artifact("outputs/decision_tree/confusion_matrix_dt.png", artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: FEATURE IMPORTANCE (O diferencial da Árvore!) ---
    importances = model_dt.feature_importances_
    fi_df = pd.DataFrame({'feature': X_train_df.columns, 'importance': importances}).sort_values(by='importance', ascending=False)
    
    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    top_fi = fi_df.head(15)
    ax_fi.barh(top_fi['feature'][::-1], top_fi['importance'][::-1], color='seagreen')
    ax_fi.set_title("Top 15 Features Mais Decisivas")
    fig_fi.savefig("outputs/decision_tree/feature_importance.png")
    mlflow.log_artifact("outputs/decision_tree/feature_importance.png", artifact_path="diagnostics")
    plt.close(fig_fi)

    # --- ARTEFATO 4: CLASSIFICATION REPORT ---
    with open("outputs/decision_tree/classification_report.txt", "w") as f:
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact("outputs/decision_tree/classification_report.txt", artifact_path="diagnostics")

    # --- ARTEFATO 5: PREPROCESSADOR E DATA ---
    joblib.dump(preprocessador, "outputs/decision_tree/preprocessador.pkl")
    mlflow.log_artifact("outputs/decision_tree/preprocessador.pkl", artifact_path="preprocessor")
    
    with open("outputs/decision_tree/feature_names.json", "w") as f:
        json.dump(list(X_train_df.columns), f)
    mlflow.log_artifact("outputs/decision_tree/feature_names.json", artifact_path="data")

   # 5. Log do Modelo PyTorch
    signature = infer_signature(X_test.head(5), y_pred[:5])

    mlflow.sklearn.log_model(
        sk_model=model_dt,
        artifact_path="modelo-arvore-decisao",
        signature=signature,
        registered_model_name="arvore-decisao-classifier-churn"
    )


print(f"Árvore de Decisão Gravada! AUC-ROC: {roc_auc:.4f} | Custo: R$ {total_cost:.2f}")



# --------------- RANDOM FOREST ---------------

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import mlflow.data
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc)
from mlflow.models.signature import infer_signature

# --- 1. CONFIGURAÇÕES E CUSTOS ---
SEED = 42
test_size = 0.2
TICKET_MEDIO = 65.0
COST_FP = TICKET_MEDIO        # R$ 65
COST_FN = TICKET_MEDIO * 5    # R$ 325
n_estimators = 100
max_depth = 10

os.makedirs("outputs/rf", exist_ok=True)

# --- 2. PREPARAÇÃO DOS DADOS ---
cols_para_remover = ['churn_value', 'churn_reason']
X = df.drop(columns=cols_para_remover)
y = df['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=SEED, stratify=y
)

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         X_train.select_dtypes(include='object').columns),
        ('num', StandardScaler(), X_train.select_dtypes(
            include=['int64', 'float64']).columns)
    ]
)

X_train_df = pd.DataFrame(preprocessador.fit_transform(
    X_train), columns=preprocessador.get_feature_names_out())
X_test_df = pd.DataFrame(preprocessador.transform(
    X_test), columns=preprocessador.get_feature_names_out())

# Dataset para o MLflow
csv_path = "data/processed/dataset_tratado.parquet"
dataset = mlflow.data.from_pandas(
    df, source=csv_path, name="dataset_normalizado_enginereed")

# --- 3. EXPERIMENTO MLFLOW ---
with mlflow.start_run(run_name="Random_Forest_Baseline_v4"):
    mlflow.log_input(dataset, context="training/test")
    # Definição do Modelo
    model_rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=SEED,
        n_jobs=-1
    )
    model_rf.fit(X_train_df, y_train)

    # Predições com Threshold de 0.30 (Consistência com os outros modelos)
    y_probs = model_rf.predict_proba(X_test_df)[:, 1]
    threshold = 0.50
    y_pred = (y_probs >= threshold).astype(int)

    # Cálculos de Métricas
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    fpr, tpr, _ = roc_curve(y_test, y_probs)
    roc_auc = auc(fpr, tpr)

    cm = confusion_matrix(y_test, y_pred)
    total_cost = (cm[0, 1] * COST_FP) + (cm[1, 0] * COST_FN)

    acc_ci = bootstrap_ci(y_test.values, y_pred, accuracy_score)
    f1_ci = bootstrap_ci(y_test.values, y_pred, f1_score)
    prec_ci = bootstrap_ci(y_test.values, y_pred, precision_score)
    rec_ci = bootstrap_ci(y_test.values, y_pred, recall_score)
    

    # Log de Tags
    mlflow.set_tags({
        "model_type": "random-forest",
        "model_architecture": "random-forest",
        "framework": "sklearn",
        "phase": "baseline",
        "dataset_name": "dataset_normalizado_enginereed",
        "dataset_size": "7043",
        "feature_set": "v2-clear-engineered"
    })

    mlflow.log_params({
        "model.model_type": "RandomForestClassifier",
        "data.n_estimators": n_estimators,
        "model.max_depth": max_depth,
        "data.random_state": SEED,
        "data.test_size": test_size,
        "data.stratify": True,
        "train.train_samples": len(X_train),
        "test.test_samples": len(X_test),
        "threshold.value": threshold,
        "business.ticket_medio": TICKET_MEDIO,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

    mlflow.log_metrics({
        "test.accuracy": acc,
        "test.f1_score": f1,
        "test.precision": prec,
        "test.recall": rec,
        "test.roc_auc": roc_auc,
        "business.cost_fp": COST_FP,
        "business.cost_fn": COST_FN,
        "business.total_cost": total_cost
    })

    # --- ARTEFATO 1: CURVA ROC ---
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    ax_roc.plot(fpr, tpr, color='rebeccapurple',
                lw=2, label=f'AUC = {roc_auc:.2f}')
    ax_roc.plot([0, 1], [0, 1], color='navy', linestyle='--')
    ax_roc.set_title("Curva ROC - Random Forest")
    ax_roc.legend()
    fig_roc.savefig("outputs/rf/roc_curve_rf.png")
    mlflow.log_artifact("outputs/rf/roc_curve_rf.png",
                        artifact_path="diagnostics")
    plt.close(fig_roc)

    # --- ARTEFATO 2: MATRIZ DE CONFUSÃO ---
    fig_cm, ax_cm = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[
                           'Fica', 'Churn']).plot(ax=ax_cm, cmap='Purples')
    ax_cm.set_title(f'Matriz de Confusão RF (Threshold {threshold})')
    fig_cm.savefig("outputs/rf/confusion_matrix_rf.png")
    mlflow.log_artifact("outputs/rf/confusion_matrix_rf.png",
                        artifact_path="diagnostics")
    plt.close(fig_cm)

    # --- ARTEFATO 3: FEATURE IMPORTANCE ---
    importances = model_rf.feature_importances_
    fi_df = pd.DataFrame({'feature': X_train_df.columns, 'importance': importances}).sort_values(
        by='importance', ascending=False)

    fig_fi, ax_fi = plt.subplots(figsize=(10, 8))
    top_fi = fi_df.head(15)
    ax_fi.barh(top_fi['feature'][::-1], top_fi['importance']
               [::-1], color='rebeccapurple')
    ax_fi.set_title("Top 15 Features - Random Forest")
    fig_fi.savefig("outputs/rf/feature_importance_rf.png")
    mlflow.log_artifact("outputs/rf/feature_importance_rf.png",
                        artifact_path="diagnostics")
    plt.close(fig_fi)

    # --- ARTEFATO 4: CLASSIFICATION REPORT ---
    with open("outputs/rf/classification_report.txt", "w") as f:
        f.write(classification_report(y_test, y_pred))
    mlflow.log_artifact("outputs/rf/classification_report.txt",
                        artifact_path="diagnostics")

    # --- ARTEFATO 5: PREPROCESSADOR E DATA ---
    joblib.dump(preprocessador, "outputs/rf/preprocessador.pkl")
    mlflow.log_artifact("outputs/rf/preprocessador.pkl",
                        artifact_path="preprocessor")

    with open("outputs/rf/feature_names.json", "w") as f:
        json.dump(list(X_train_df.columns), f)
    mlflow.log_artifact("outputs/rf/feature_names.json", artifact_path="data")

    # 5. Log do Modelo PyTorch
    signature = infer_signature(X_test.head(5), y_pred[:5])

    mlflow.sklearn.log_model(
        sk_model=model_rf,
        artifact_path="modelo-random-forest",
        signature=signature,
        registered_model_name="random-forest-classifier-churn"
    )

print(
    f"Random Forest Gravado! AUC-ROC: {roc_auc:.4f} | Custo Total: R$ {total_cost:.2f}")


c:\Python311\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Python311\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-in

Random Forest Gravado! AUC-ROC: 0.8384 | Custo Total: R$ 81640.00

📊 TABELA FINAL COM INTERVALO DE CONFIANÇA:


,model,accuracy,accuracy_ci,recall,recall_ci,precision,precision_ci,f1_score,f1_ci,roc_auc,total_cost
0,Random Forest,0.789922,"[0.770, 0.811]",0.358289,"[0.311, 0.406]",0.705263,"[0.642, 0.770]",0.475177,"[0.425, 0.524]",0.838352,81640.0


========================= COMPARACAO DE METRICAS ENTRE MODELOS + ANALISE DE CUSTOS ==========================

In [5]:
# --- CONFIGURAÇÕES DO EXPERIMENTO ---
experiment_name = "Tech-Challenge-Telco-Churn"
experiment = mlflow.get_experiment_by_name(experiment_name)
df_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

df_clean = df_runs[["tags.model_type","metrics.test.recall", "metrics.test.precision", "metrics.test.roc_auc","metrics.test.f1_score","metrics.test.accuracy","metrics.business.total_cost"]]
df_clean.columns = [c.replace("test.", "") for c in df_clean.columns]
leaderboard = df_clean.sort_values("metrics.business.total_cost", ascending=True).drop_duplicates("tags.model_type")

cols_finais = [
    "tags.model_type",
    "metrics.accuracy", 
    "metrics.recall", 
    "metrics.precision", 
    "metrics.roc_auc", 
    "metrics.f1_score",
    "metrics.business.total_cost"
]

leaderboard = leaderboard[cols_finais].sort_values("metrics.business.total_cost")
leaderboard_fmt = leaderboard.copy()

leaderboard_fmt["metrics.business.total_cost"] = leaderboard_fmt["metrics.business.total_cost"].apply(lambda x: f"R$ {x:,.2f}")
display(leaderboard_fmt)

,tags.model_type,metrics.accuracy,metrics.recall,metrics.precision,metrics.roc_auc,metrics.f1_score,metrics.business.total_cost
17,mlp_neural_network,0.665720,0.927807,0.438685,0.847358,0.595708,"R$ 37,635.00"
4,random-forest,0.756565,0.799465,0.527337,0.838352,0.635494,"R$ 41,795.00"
7,arvore_de_decisao,0.745209,0.791444,0.512998,0.833534,0.622503,"R$ 43,615.00"
15,regressao_logistica,0.755855,0.748663,0.528302,0.843509,0.619469,"R$ 46,800.00"
11,dummy_classifier,0.604684,0.245989,0.250681,0.490144,0.248313,"R$ 109,525.00"
